Домашнее задание
Симуляция распространения болезни. Работа с NetworkX

Цель:
смоделировать распространение инфекционного заболевания в международной сети аэропортов с использованием методов анализа графов;
проанализировать результаты моделирования и оценить влияние различных аэропортов на скорость и масштаб распространения инфекции.


Описание/Пошаговая инструкция выполнения домашнего задания:
Набор данных представляет собой основные маршруты авиаперелетов в США за один месяц 2008-го года. В качестве модели распространения инфекции предлагается использовать SI (susceptible-infected) модель, в которой каждый аэропорт имеет два возможных состояния - “здоровый” или “зараженный”. Здоровый аэропорт может заразиться с некоторой вероятностью p (infection probability), если к нему прилетает самолет из зараженного города. Инфекция начинает распространение из некоторого стартового аэропорта и заканчивает распространение, когда все перелеты из датасета завершены.


Часть 1. Пишем симуляцию.

Напишите функцию для симуляции одного прохода по всему датасету и распространения инфекции. Функция должна принимать на вход стартовый город, в котором изначально началась инфекция и вероятность распространения инфекции p. На выходе функция должна возвращать словарь, в котором ключом будет время заражения аэропорта, значением - название аэропорта. В теле функции вам нужно написать проход по всему датасету (имеет смысл использовать itertuples), где вы будете проверять, является ли текущий рейс “заразным” и если да - с вероятностью p заражать новый аэропорт.
В качестве стартовой ноды/аэропорта для симуляций можно использовать Allentown (node_id = 0).

Часть 2. Как инфекционность (вероятность заражения) болезни влияет на скорость распространения?

Давайте посмотрим, насколько заразной должна быть болезнь, чтобы заразить все аэропорты в сети. Для каждой из вероятностей p=[0.01, 0.05, 0.1, 0.5, 1.] прогоните по 10 симуляций распространения, каждый раз сохраняя результаты.
Для каждых 12 часов симуляции посчитайте, какой средний процент аэропортов от общего их числа, был заражен к этому моменту времени. Постройте графики числа зараженных городов в зависимости от времени для каждой из вероятностей.

Часть 3. Может ли информация о сети помочь предсказать, кто будет заражен?

Настало время поработать с NetworkX. По имеющемуся датасету вам нужно построить ненаправленный граф всех аэропортов, используя относительное число перелетов между соседними аэропортами в качестве веса ребра. Например, если из аэропорта А в аэропорт Б суммарно есть 5 рейсов, из Б в А 10 рейсов, а всего в сети 150 перелетов, то вес ребра между А и Б должен быть равен (5+10)/150 = 0.1.
Прогоните вашу симуляцию 50 раз, используя p=0.5 и стартовый аэропорт Allentown, каждый раз сохраняя результаты. Используя данные симуляций посчитайте медианное время до заражения каждого города.
Теперь используя построенный вами граф, для каждого города посчитайте его коэффициент кластеризации (nx.clustering), степень (nx.degree) и центральность (nx.betweenness_centrality).
Наконец, постройте scatter-plot-ы для медианного времени заражения и каждой из рассчитанных выше метрик. Также посчитайте коэффициент корреляции Спирмана между этими показателями и попробуйте проинтерпретировать полученные результаты. Какая из метрик графа сильнее всего скоррелирована со временем заражения? Почему?

https://www.kaggle.com/datasets/vikalpdongre/us-flights-data-2008

Критерии оценки:
Если вы добрались до этой задачки - уже 2 балла.
Написана работающая функция для симуляции - 2 балла.
Протестировано влияние вероятности заражения на скорость распространения - 3 балла.
Построены scatter-plot-ы метрик графа и медианного времени заражения + интерпретация результатов - 3 балла.

In [2]:
# # to load df from cloud
# import kagglehub
#
# # Download latest version
# path = kagglehub.dataset_download("vikalpdongre/us-flights-data-2008")
#
# print("Path to dataset files:", path)

In [15]:
# =====================
# 1. CSV load and setup configs
# =====================

import os
import random
from collections import deque, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import spearmanr
import plotly.express as px

RANDOM_STATE = 555
pd.set_option('display.max_columns', None)

base_dir = os.getcwd()
input_dir = os.path.join(base_dir, 'csv_input')
output_dir = os.path.join(base_dir, 'csv_output')
os.makedirs(output_dir, exist_ok=True)
results_path = os.path.join(output_dir, 'model_results.csv')

START_AIRPORT = 'CMX'  # initial start point 'ABE' Allentown, node_id = 0

def log_result(row):
    # append evaul row into metrics log
    row = {'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'), **row}
    pd.DataFrame([row]).to_csv(results_path, mode='a', header=not os.path.exists(results_path), index=False)

# Making separate file with January 2008 only
source_path = os.path.join(input_dir, '2008.csv')
file_path = os.path.join(input_dir, '2008_month1.csv')

start_row = 0
end_row = 605800

if not os.path.exists(file_path):
    print('Reading rows', start_row, 'to', end_row, 'from:', source_path)
    df_month1 = pd.read_csv(source_path, skiprows=range(1, start_row), nrows=end_row - start_row + 1)
    print('Rows read:', len(df_month1))
    print('Month distribution in the read range:')
    print(df_month1['Month'].value_counts().sort_index().to_string())
    n_before = len(df_month1)
    df_month1 = df_month1[df_month1['Month'] == 1].reset_index(drop=True)
    print('Rows kept for Month = 1:', len(df_month1), 'of', n_before)
    print('Days:', int(df_month1['DayofMonth'].min()), '-', int(df_month1['DayofMonth'].max()))
    df_month1.to_csv(file_path, index=False)
    print('Saved:', file_path)

# keep columns for simulation only
all_cols = pd.read_csv(file_path, nrows=0).columns.tolist()
exclude_cols = [c for c in all_cols if c not in ('DayofMonth', 'DepTime', 'ArrTime', 'Origin', 'Dest', 'Cancelled', 'Diverted')]
use_cols = [col for col in all_cols if col not in exclude_cols]

df_raw = pd.read_csv(file_path, usecols=use_cols)
print('Dataset shape (January 2008):', df_raw.shape)
df_raw.head()

Dataset shape (January 2008): (605765, 7)


,DayofMonth,DepTime,ArrTime,Origin,Dest,Cancelled,Diverted
0,3,2003.0,2211.0,IAD,TPA,0,0
1,3,754.0,1002.0,IAD,TPA,0,0
2,3,628.0,804.0,IND,BWI,0,0
3,3,926.0,1054.0,IND,BWI,0,0
4,3,1829.0,1959.0,IND,BWI,0,0


In [16]:
#--------------------
# 2. Cleaning and building flight timeline
#--------------------

n_total = len(df_raw)
df_flights = df_raw[(df_raw['Cancelled'] == 0) & (df_raw['Diverted'] == 0)].copy()
df_flights = df_flights.dropna(subset=['DepTime', 'ArrTime', 'Origin', 'Dest'])

# departure and arrival time in minutes from the start of the month window (day 1, 00:00)
hhmm_dep = df_flights['DepTime'].astype('int64')
hhmm_arr = df_flights['ArrTime'].astype('int64')
df_flights['dep_time_min'] = (df_flights['DayofMonth'] - 1) * 24 * 60 + (hhmm_dep // 100) * 60 + hhmm_dep % 100
df_flights['arr_time_min'] = (df_flights['DayofMonth'] - 1) * 24 * 60 + (hhmm_arr // 100) * 60 + hhmm_arr % 100

# overnight flights (arrival hhmm < departure hhmm) -> next day
overnight = df_flights['arr_time_min'] < df_flights['dep_time_min']
df_flights.loc[overnight, 'arr_time_min'] += 24 * 60
print('Overnight flights (arrival next day):', int(overnight.sum()))

# Simulation uses the timeline in arrival order
df_flights = df_flights.sort_values('arr_time_min', kind='mergesort').reset_index(drop=True)

print('Rows in raw dataset:', n_total)
print('Rows dropped (cancelled, diverted, missing times):', n_total - len(df_flights))
print('Departure time span, minutes:', int(df_flights['dep_time_min'].min()), '-', int(df_flights['dep_time_min'].max()))
print('Arrival time span, minutes:', int(df_flights['arr_time_min'].min()), '-', int(df_flights['arr_time_min'].max()))

# tuple list reused by every simulation
df_flights['Origin'] = df_flights['Origin'].astype('category')
df_flights['Dest'] = df_flights['Dest'].astype('category')
flights = [(t_arr, t_dep, o, d)
           for t_arr, t_dep, o, d in df_flights[['arr_time_min', 'dep_time_min', 'Origin', 'Dest']].itertuples(index=False, name=None)]
print('Flights in the simulation timeline:', len(flights))

all_airports = sorted(set(df_flights['Origin'].cat.categories) | set(df_flights['Dest'].cat.categories))
print('Airports in the network:', len(all_airports))

Overnight flights (arrival next day): 17673
Rows in raw dataset: 605765
Rows dropped (cancelled, diverted, missing times): 18635
Departure time span, minutes: 1 - 44640
Arrival time span, minutes: 57 - 46003
Flights in the simulation timeline: 587130
Airports in the network: 286


In [17]:
#--------------------
# 3 EDA of the flight network
#--------------------

n_days = df_flights['DayofMonth'].nunique()
print('Total flights in the timeline:', len(df_flights))
print('Airports in the network:', len(all_airports))
print('Days covered:', n_days)
print('Average flights per day:', round(len(df_flights) / n_days, 4))

top_airports = df_flights['Origin'].value_counts().head(10)
fig = px.bar(x=top_airports.index.astype(str), y=top_airports.values,
             labels={'x': 'airport', 'y': 'departures'},
             title='Top 10 airports by number of departures')
fig.update_yaxes(range=[0, int(np.ceil(top_airports.max() / 10000) * 10000)])
fig.update_layout(width=800, height=450)
fig.show()

Total flights in the timeline: 587130
Airports in the network: 286
Days covered: 31
Average flights per day: 18939.6774


In [18]:
#--------------------
# 4. Simulation: one pass over the whole dataset
#--------------------

def simulate_si(start_airport, p, flights):
    # flight is infection when its origin already infected before the flight departure,
    # Destination city is infected at the arrival moment with probability 'p'
    # returns a dict {infection_time_min: [airports infected at that minute]}
    inf_time = {start_airport: 0}
    by_time = {0: [start_airport]}
    never = 10 ** 9
    rnd = random.random
    for t_arr, t_dep, origin, dest in flights:
        if inf_time.get(origin, never) <= t_dep and dest not in inf_time and rnd() < p:
            inf_time[dest] = t_arr
            by_time.setdefault(t_arr, []).append(dest)
    return by_time

def to_airport_times(by_time):
    # convert {time: [airports]} to {airport: infection_time}
    return {apt: t for t, apts in by_time.items() for apt in apts}

random.seed(RANDOM_STATE)
demo_by_time = simulate_si(START_AIRPORT, 0.5, flights)
demo_times = to_airport_times(demo_by_time)
print('Start airport:', START_AIRPORT, '(Allentown, node_id = 0)')
print('Infected airports:', len(demo_times), 'of', len(all_airports),
      '(share', round(len(demo_times) / len(all_airports), 4), ')')
first_infected = sorted(demo_times.items(), key=lambda kv: kv[1])[:6]
print('First infected airports:', first_infected)
last_infected = sorted(demo_times.items(), key=lambda kv: kv[1])[-6:]
print('Last infected airports:', last_infected)

Start airport: CMX (Allentown, node_id = 0)
Infected airports: 286 of 286 (share 1.0 )
First infected airports: [('CMX', 0), ('MSP', 460), ('STL', 558), ('BIS', 617), ('MLI', 621), ('LAS', 631)]
Last infected airports: [('CSG', 3500), ('ADK', 3859), ('GTR', 5028), ('ALO', 5726), ('PLN', 10049), ('CDC', 11368)]


In [19]:
#--------------------
# 5 Part 2 - Infection probability vs speed of spread
#--------------------

p_values = [0.01, 0.05, 0.1, 0.5, 1.0]
n_sims = 10
window_min = 12 * 60
t_max = int(df_flights['arr_time_min'].max())
window_ends = np.arange(window_min, t_max + window_min, window_min, dtype='int64')
n_airports_total = len(all_airports)

def cumulative_infected(times, window_ends):
    # Number of infected airports by the end of each 12h window.
    arr = np.sort(np.asarray(list(times), dtype='int64'))
    return np.searchsorted(arr, window_ends, side='right')

sweep_rows = []
curves = {}
random.seed(RANDOM_STATE)
for p in p_values:
    sims_curves = []
    final_counts = []
    for _ in range(n_sims):
        airport_times = to_airport_times(simulate_si(START_AIRPORT, p, flights))
        sims_curves.append(cumulative_infected(airport_times.values(), window_ends))
        final_counts.append(len(airport_times))
    avg_curve = np.mean(sims_curves, axis=0)
    curves[p] = avg_curve

    row = {'p': p,
           'infected_final': round(float(np.mean(final_counts)), 4),
           'infected_share_final': round(float(np.mean(final_counts)) / n_airports_total, 4)}
    for share_level, col in ((0.5, 'window_h_50pct'), (0.9, 'window_h_90pct'), (1.0, 'window_h_100pct')):
        reached = avg_curve >= n_airports_total * share_level
        row[col] = round(float(window_ends[np.argmax(reached)]) / 60, 4) if reached.any() else None
    sweep_rows.append(row)
    print('p =', p, ': infected', row['infected_final'], 'of', n_airports_total,
          '(share', row['infected_share_final'], '), 50% at', row['window_h_50pct'],
          'h, 90% at', row['window_h_90pct'], 'h, 100% at', row['window_h_100pct'], 'h')

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(os.path.join(output_dir, 'part2_p_sweep.csv'), index=False)
sweep_df

p = 0.01 : infected 32.7 of 286 (share 0.1143 ), 50% at None h, 90% at None h, 100% at None h
p = 0.05 : infected 156.4 of 286 (share 0.5469 ), 50% at 696.0 h, 90% at None h, 100% at None h
p = 0.1 : infected 135.7 of 286 (share 0.4745 ), 50% at None h, 90% at None h, 100% at None h
p = 0.5 : infected 285.8 of 286 (share 0.9993 ), 50% at 72.0 h, 90% at 240.0 h, 100% at None h
p = 1.0 : infected 286.0 of 286 (share 1.0 ), 50% at 24.0 h, 90% at 24.0 h, 100% at 168.0 h


,p,infected_final,infected_share_final,window_h_50pct,window_h_90pct,window_h_100pct
0,0.01,32.7,0.1143,NaN,NaN,NaN
1,0.05,156.4,0.5469,696.0,NaN,NaN
2,0.10,135.7,0.4745,NaN,NaN,NaN
3,0.50,285.8,0.9993,72.0,240.0,NaN
4,1.00,286.0,1.0000,24.0,24.0,168.0


In [20]:
# average infected count per 12h window for every p (mean over 10 simulations)
plot_rows = []
for p in p_values:
    for w_end, cnt in zip(window_ends, curves[p]):
        plot_rows.append({'p': str(p), 'time_h': round(w_end / 60, 4), 'infected': round(float(cnt), 4)})
sweep_curves_df = pd.DataFrame(plot_rows)
sweep_curves_df.to_csv(os.path.join(output_dir, 'part2_infected_by_12h.csv'), index=False)

fig = px.line(sweep_curves_df, x='time_h', y='infected', color='p',
              labels={'time_h': 'time, hours', 'infected': 'infected airports'},
              title='Average number of infected airports by 12h window (10 simulations per p)')
fig.update_yaxes(range=[0, n_airports_total])
fig.update_layout(width=900, height=500)
fig.show()

for row in sweep_rows:
    log_result({'model': 'SI simulation p sweep',
                'hyperparameters': 'start=' + START_AIRPORT + ', p=' + str(row['p']) + ', n_sims=' + str(n_sims),
                'infected_share_final': row['infected_share_final'],
                'window_h_50pct': row['window_h_50pct'],
                'window_h_90pct': row['window_h_90pct'],
                'window_h_100pct': row['window_h_100pct']})
print('Results logged to', results_path)

Results logged to E:\py_otus\ml-pro\homework_02_13\csv_output\model_results.csv


In [21]:
#--------------------
# 6. Part 3. Undirected airport graph: edge weights, DFS components, BFS hops
#--------------------

total_flights = len(df_flights)

# edge weight = (flights A->B + flights B->A) / total flights in the network
pair_counts = defaultdict(int)
for (o, d), c in df_flights.groupby(['Origin', 'Dest'], observed=True).size().items():
    key = (o, d) if o <= d else (d, o)
    pair_counts[key] += c

G = nx.Graph()
for (a, b), c in pair_counts.items():
    G.add_edge(a, b, weight=round(c / total_flights, 4))
print('Graph:', G.number_of_nodes(), 'nodes,', G.number_of_edges(), 'edges')
print('Sum of edge weights (total share of flights covered by the undirected graph):', round(sum(w['weight'] for _, _, w in G.edges(data=True)), 4))

def dfs_components(graph):
    # Iterative depth-first search: connected components of the undirected graph.
    visited = set()
    components = []
    for start in graph.nodes:
        if start in visited:
            continue
        component = []
        stack = [start]
        visited.add(start)
        while stack:
            node = stack.pop()
            component.append(node)
            for neighbor in graph.neighbors(node):
                if neighbor not in visited:
                    visited.add(neighbor)
                    stack.append(neighbor)
        components.append(component)
    return components

components = sorted(dfs_components(G), key=len, reverse=True)
print('Connected components (DFS):', len(components))
print('Component sizes:', [len(c) for c in components])
start_component = next(c for c in components if START_AIRPORT in c)
print('Component of the start airport', START_AIRPORT, ':', len(start_component), 'airports')

def bfs_hops(graph, source):
    # minimum number of flight legs from source to every reachable airport.
    dist = {source: 0}
    queue = deque([source])
    while queue:
        node = queue.popleft()
        for neighbor in graph.neighbors(node):
            if neighbor not in dist:
                dist[neighbor] = dist[node] + 1
                queue.append(neighbor)
    return dist

hops_from_start = bfs_hops(G, START_AIRPORT)
print('Airports reachable from', START_AIRPORT, '(BFS):', len(hops_from_start), 'of', G.number_of_nodes())
print('Max number of legs from the start airport:', max(hops_from_start.values()))
print('Airports unreachable in the static graph:', sorted(set(G.nodes) - set(hops_from_start)))

Graph: 286 nodes, 2319 edges
Sum of edge weights (total share of flights covered by the undirected graph): 1.0009
Connected components (DFS): 1
Component sizes: [286]
Component of the start airport CMX : 286 airports
Airports reachable from CMX (BFS): 286 of 286
Max number of legs from the start airport: 4
Airports unreachable in the static graph: []


In [22]:
#--------------------
# 7. Part 3. Median time to infection (50 sims)
#--------------------

n_sims_part3 = 50
p_part3 = 0.5

infection_runs = defaultdict(list)
random.seed(RANDOM_STATE)
for _ in range(n_sims_part3):
    airport_times = to_airport_times(simulate_si(START_AIRPORT, p_part3, flights))
    for apt, t in airport_times.items():
        infection_runs[apt].append(t)

clustering = nx.clustering(G)
degrees = dict(nx.degree(G))
betweenness = nx.betweenness_centrality(G)

df_result = pd.DataFrame(index=sorted(G.nodes))
df_result.index.name = 'airport'
df_result['n_infected_runs'] = [len(infection_runs.get(a, [])) for a in df_result.index]
df_result['median_time'] = [round(float(np.median(infection_runs[a])), 4) if a in infection_runs else np.nan
                            for a in df_result.index]
df_result['degree'] = [degrees[a] for a in df_result.index]
df_result['clustering'] = [round(clustering[a], 4) for a in df_result.index]
df_result['betweenness'] = [round(betweenness[a], 4) for a in df_result.index]
df_result['bfs_hops'] = [hops_from_start.get(a) for a in df_result.index]

print('Airports never infected in', n_sims_part3, 'runs:', int((df_result['n_infected_runs'] == 0).sum()))
print('Airports infected in all', n_sims_part3, 'runs:', int((df_result['n_infected_runs'] == n_sims_part3).sum()))
df_result.to_csv(os.path.join(output_dir, 'part3_airport_metrics.csv'))
df_result.head(10)

Airports never infected in 50 runs: 0
Airports infected in all 50 runs: 285


,n_infected_runs,median_time,degree,clustering,betweenness,bfs_hops
airport,,,,,,
ABE,50,1153.5,7,1.0000,0.0000,3
ABI,50,1196.0,1,0.0000,0.0000,3
ABQ,50,876.0,37,0.6712,0.0025,2
ABY,50,3494.0,1,0.0000,0.0000,3
ACT,50,1657.0,1,0.0000,0.0000,3
ACV,50,1106.0,3,0.6667,0.0000,3
ACY,50,4253.0,2,1.0000,0.0000,3
ADK,50,8187.0,1,0.0000,0.0000,3
ADQ,50,3934.0,1,0.0000,0.0000,3


In [24]:
#--------------------
# 8. Part 3. Median infection time vs graph metrics: scatter plots and Spearman corr
#--------------------

metric_cols = ['degree', 'clustering', 'betweenness', 'bfs_hops']
labels = {'degree': 'degree', 'clustering': 'clustering coefficient',
          'betweenness': 'betweenness centrality', 'bfs_hops': 'BFS hops from start'}

spearman_rows = []
for metric in metric_cols:
    sub = df_result[['median_time', metric]].dropna()
    rho, p_value = spearmanr(sub[metric], sub['median_time'])
    rho, p_value = round(float(rho), 4), round(float(p_value), 4)
    spearman_rows.append({'metric': metric, 'spearman_rho': rho, 'p_value': p_value})
    print(labels[metric], ': Spearman rho =', rho, '(p-value =', p_value, ')')
    fig = px.scatter(sub, x=metric, y='median_time',
                     labels={metric: labels[metric], 'median_time': 'median infection time, min'},
                     title='Median infection time vs ' + labels[metric] + ' (Spearman rho = ' + str(rho) + ')')
    fig.update_layout(width=700, height=450)
    fig.show()

spearman_df = pd.DataFrame(spearman_rows)
strongest = spearman_df.loc[spearman_df['spearman_rho'].abs().idxmax()]
print('Strongest correlation with median infection time:', strongest['metric'],
      'rho =', strongest['spearman_rho'])

log_row = {'model': 'SI graph analysis',
           'hyperparameters': 'start=' + START_AIRPORT + ', p=' + str(p_part3) + ', n_sims=' + str(n_sims_part3)}
for r in spearman_rows:
    log_row['spearman_' + r['metric']] = r['spearman_rho']
log_result(log_row)


degree : Spearman rho = -0.8637 (p-value = 0.0 )


clustering coefficient : Spearman rho = -0.1267 (p-value = 0.0321 )


betweenness centrality : Spearman rho = -0.6229 (p-value = 0.0 )


BFS hops from start : Spearman rho = 0.5879 (p-value = 0.0 )


Strongest correlation with median infection time: degree rho = -0.8637


In [25]:
results_df = pd.read_csv(results_path)
results_df = results_df.drop_duplicates(subset=['model', 'hyperparameters'], keep='last').reset_index(drop=True)
results_df

,timestamp,model,hyperparameters,infected_share_final,window_h_50pct,window_h_90pct,window_h_100pct
0,2026-10-27 23:18:31,SI simulation p sweep,"start=ABE, p=0.01, n_sims=10",0.7979,312.0000,NaN,NaN
1,2026-10-27 23:18:31,SI simulation p sweep,"start=ABE, p=0.05, n_sims=10",0.9731,72.0000,288.0000,NaN
2,2026-10-27 23:18:31,SI simulation p sweep,"start=ABE, p=0.1, n_sims=10",0.9930,48.0000,156.0000,NaN
3,2026-10-27 23:18:31,SI simulation p sweep,"start=ABE, p=0.5, n_sims=10",1.0000,24.0000,48.0000,576.0000
4,2026-10-27 23:18:31,SI simulation p sweep,"start=ABE, p=1.0, n_sims=10",1.0000,24.0000,24.0000,168.0000
5,2026-10-27 23:20:28,SI graph analysis,"start=ABE, p=0.5, n_sims=50",-0.8073,-0.1824,-0.5408,0.5000
6,2026-10-27 23:30:28,SI simulation p sweep,"start=CMX, p=0.01, n_sims=10",0.1143,NaN,NaN,NaN
7,2026-10-27 23:30:28,SI simulation p sweep,"start=CMX, p=0.05, n_sims=10",0.5469,696.0000,NaN,NaN
8,2026-10-27 23:30:28,SI simulation p sweep,"start=CMX, p=0.1, n_sims=10",0.4745,NaN,NaN,NaN
9,2026-10-27 23:30:28,SI simulation p sweep,"start=CMX, p=0.5, n_sims=10",0.9993,72.0000,240.0000,NaN


In [ ]:
# =====================
# 9. Conclusions
# =====================
#
# Для уменьшения объема данных взял январь месяц из основного датасета. посмотрел где примерно начинается месяц 2, убедился что данные отсортированы по месяцам. сделал отдельный датасет на январь.
# оставил основные столбцы для ускорения расчетов
# ('DayofMonth', 'DepTime', 'ArrTime', 'Origin', 'Dest', 'Cancelled', 'Diverted')

# попробовал разные стартовые точки
# 'CMX' при p=0.01 конечно много времени прошло до второго заражения, всего за 708 часов получилось 27.7 точек,
# при p=0.1 129.4 точки. по графику видно что заражение происходит скачкообразно.

# В общем симуляция из города ABE показала быстрое распростронение даже для средней вероятности. Пробовал другие стартовые города, результаты разные. Зависит от того как быстро инфекция попадает в аэропорты с большим кол рейсов

# по пункту 5 (сим часть 2) видно, что распространение происходит скачками, ну и чем раньше сработала вероятность, тем быстрее происходит скачок. Т.е. больша вероятность вируса гарантирует ранние скачки и большего масштаба (больше рейсов успешно переносят вирус из крупного узлового города
#
# Еще важный параметр - время рейса. Т.е. из рейсы, где перелет быстрый, заражают быстрее :)
# по поиску в ширину получается что сама длина маршрута (по аэропортам) - значимый показатель, каждая дополнительная ветка маршрута из ABE ведет к дополнительной задержке


